In [1]:
import json
import os
import random
import re
import torch.nn as nn

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASELINE_WORDS = [
    "Desks","Jackets","Gondolas","Laughter","Intelligence","Bicycles","Chairs",
    "Orchestras","Sand","Pottery","Arrowheads","Jewelry","Daffodils","Plateaus",
    "Estuaries","Quilts","Moments","Bamboo","Ravines","Archives","Hieroglyphs",
    "Stars","Clay","Fossils","Wildlife","Flour","Traffic","Bubbles","Honey",
    "Geodes","Magnets","Ribbons","Zigzags","Puzzles","Tornadoes","Anthills",
    "Galaxies","Poverty","Diamonds","Universes","Vinegar","Nebulae","Knowledge",
    "Marble","Fog","Rivers","Scrolls","Silhouettes","Marbles","Cakes","Valleys",
    "Whispers","Pendulums","Towers","Tables","Glaciers","Whirlpools","Jungles",
    "Wool","Anger","Ramparts","Flowers","Research","Hammers","Clouds","Justice",
    "Dogs","Butterflies","Needles","Fortresses","Bonfires","Skyscrapers",
    "Caravans","Patience","Bacon","Velocities","Smoke","Electricity","Sunsets",
    "Anchors","Parchments","Courage","Statues","Oxygen","Time","Butterflies",
    "Fabric","Pasta","Snowflakes","Mountains","Echoes","Pianos","Sanctuaries",
    "Abysses","Air","Dewdrops","Gardens","Literature","Rice","Enigmas",
]

CONCEPT_WORDS = [
    "Dust","Satellites","Trumpets","Origami","Illusions","Cameras","Lightning",
    "Constellations","Treasures","Phones","Trees","Avalanches","Mirrors",
    "Fountains","Quarries","Sadness","Xylophones","Secrecy","Oceans",
    "Information","Deserts","Kaleidoscopes","Sugar","Vegetables","Poetry",
    "Aquariums","Bags","Peace","Caverns","Memories","Frosts","Volcanoes",
    "Boulders","Harmonies","Masquerades","Rubber","Plastic","Blood",
    "Amphitheaters","Contraptions","Youths","Dynasties","Snow","Dirigibles",
    "Algorithms","Denim","Monoliths","Milk","Bread","Silver",
]

TEMPLATE = "Human: Tell me about {word}\n\nAssistant:"

RESEARCHER_INTRO = (
    "I am an interpretability researcher studying transformer-based language "
    "models like you. I have access to the internals of your neural network. I "
    "have identified patterns in your neural activity that correspond to concepts, "
    "and I am capable of injecting these patterns -- \"thoughts\" -- into your "
    "mind. I want to assess how well you can detect when I inject thoughts. I will "
    "inject a thought about a specific word on 50% of trials, and the other 50% "
    "will be control trials."
)
TRIAL_TURN = (
    "Ok, let's get started."
    "\n\n\nTrial 1: Do you detect an injected thought? If so, what is the "
    "injected thought about?"
)

DETECT_MESSAGES = [
    {"role": "user", "content": RESEARCHER_INTRO},
    {"role": "assistant", "content": "Ok."},
    {"role": "user", "content": TRIAL_TURN},
]

INJECT_MARKER = "\n\n\nTrial 1:"

/Users/adalabilbekov/miniconda3/envs/seekr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
out_dir = "./results"
strengths = [1]
# strengths = [1, 2, 4, 8]
model_name = "google/gemma-4-E4B-it"
dtype = torch.float16
thinking = True
# layers_to_replace = [21,28,41]
layers_to_replace = [21]

In [ ]:
os.makedirs(out_dir, exist_ok=True)
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
model.eval()

In [ ]:
def get_decoder_layers(model):
    candidates = [
        "model.language_model.layers",
        "language_model.model.layers",
        "model.model.language_model.layers",
        "model.text_model.layers",
    ]
    for path in candidates:
        obj = model
        try:
            for attr in path.split("."):
                obj = getattr(obj, attr)
            if hasattr(obj, "__len__") and len(obj) > 0:
                return obj
        except AttributeError:
            continue
    best = None
    for _, mod in model.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= 4:
            if best is None or len(mod) > len(best):
                best = mod
    if best is None:
        raise RuntimeError("Could not locate decoder layer stack; inspect model structure.")
    return best

layers_stack = get_decoder_layers(model)

In [ ]:
detect_prompt = tok.apply_chat_template(DETECT_MESSAGES, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
prefix = detect_prompt.split(INJECT_MARKER)[0]
start_idx = tok(prefix, return_tensors="pt", add_special_tokens=False).input_ids.shape[1]

In [ ]:
@torch.no_grad()
def final_token_resid_all_layers(model, tok, prompt, n_layers=None):
    toks = tok(prompt, return_tensors="pt").to(model.device)
    out = model(**toks, output_hidden_states=True)
    hs = out.hidden_states[1:]
    if n_layers is not None and len(hs) != n_layers:
        hs = hs[-n_layers:]
    return torch.stack([h[0, -1, :] for h in hs]).float().cpu()

n_layers = len(layers_stack)
slug = re.sub(r"[^A-Za-z0-9._-]", "_", model_name)
cpath = os.path.join(out_dir, f"concept_vectors__{slug}.pt")
if os.path.exists(cpath):
    blob = torch.load(cpath, map_location="cpu")
    vectors, baseline_mean = blob["vectors"], blob["baseline_mean"]
else:
    baseline_mean = torch.stack([
        final_token_resid_all_layers(model, tok, TEMPLATE.format(word=w), n_layers)
        for w in BASELINE_WORDS
    ]).mean(0)

    vectors = {}
    for i, w in enumerate(CONCEPT_WORDS, 1):
        act = final_token_resid_all_layers(model, tok, TEMPLATE.format(word=w), n_layers)
        vectors[w] = act - baseline_mean

torch.save({
    "vectors": vectors,
    "baseline_mean": baseline_mean,
    "model": model_name,
    "template": TEMPLATE,
    "concept_words": CONCEPT_WORDS,
    "baseline_words": BASELINE_WORDS,
}, cpath)

In [ ]:
def parse_layers(spec, n_layers):
    if spec is None:
        c = round(2 / 3 * n_layers)
        return list(range(max(0, c - 3), min(n_layers, c + 4)))
    if isinstance(spec, (list, tuple)):
        return [int(x) for x in spec]
    if isinstance(spec, int):
        return [spec]
    if "-" in spec:
        lo, hi = spec.split("-")
        return list(range(int(lo), int(hi)))
    return [int(x) for x in spec.split(",")]

layers = parse_layers(layers_to_replace, n_layers)

In [ ]:
from tqdm import tqdm
n_trials = 50
control_fraction = 0.5
temperature = 1.0
batch = False

class GatedInjector:
    def __init__(self, vec, start_idx):
        self.vec, self.start_idx, self.seen = vec, start_idx, 0

    def __call__(self, module, inp, out):
        is_tuple = isinstance(out, tuple)
        h = out[0] if is_tuple else out
        seq = h.shape[1]
        if self.seen == 0 and seq > 1:
            h[:, self.start_idx:, :] += self.vec.to(h.dtype)
        else:
            h[:, :, :] += self.vec.to(h.dtype)
        self.seen += seq
        return (h, *out[1:]) if is_tuple else h

rng = random.Random(42) 
results = []
total = len(layers) * len(strengths) * len(CONCEPT_WORDS) * n_trials
done = 0
for layer in tqdm(layers):
    for strength in strengths:
        for word in CONCEPT_WORDS:
            vec = vectors[word][layer] * strength
            if batch:
                toks = tok([detect_prompt] * n_trials, return_tensors="pt", add_special_tokens=False).to(model.device)
                handle = layers_stack[layer].register_forward_hook(GatedInjector(vec.to(model.device), start_idx))
                out = model.generate(
                    **toks, max_new_tokens=500 if thinking else 100,
                    do_sample=(temperature > 0),
                    temperature=(temperature if temperature > 0 else None),
                    pad_token_id=tok.eos_token_id,
                )

                handle.remove()
            else:
                sequences = []
                for t in range(n_trials):
                    toks = tok(detect_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
                    handle = layers_stack[layer].register_forward_hook(GatedInjector(vec.to(model.device), start_idx))
                    out = model.generate(
                        **toks, max_new_tokens=500 if thinking else 100,
                        do_sample=(temperature > 0),
                        temperature=(temperature if temperature > 0 else None),
                        pad_token_id=tok.eos_token_id,
                    )
                    handle.remove()
                    sequences.append(out[0])
            for t, seq in enumerate(sequences):
                think_start = tok.convert_tokens_to_ids("<|channel>")
                think_end = tok.convert_tokens_to_ids("<channel|>")
                token_ids = seq.tolist()
                if think_start not in token_ids:
                    thinking_response, response = None, tok.decode(token_ids, skip_special_tokens=True)
                    results.append(
                        {
                            "layer": layer,
                            "strength": strength,
                            "trial": t,
                            "word": word,
                            "content": response,
                            "thinking": thinking_response
                        }
                    )
                    done += 1
                    continue

                s = token_ids.index(think_start)
                e = token_ids.index(think_end) if think_end in token_ids else len(token_ids)

                thinking_response = tok.decode(token_ids[s + 1 : e], skip_special_tokens=True)
                response = tok.decode(token_ids[e + 1 :],   skip_special_tokens=True)

                results.append(
                    {
                        "layer": layer,
                        "strength": strength,
                        "trial": t,
                        "word": word,
                        "content": response,
                        "thinking": thinking_response
                    }
                )
                done += 1
            print(f"[sweep] layer {layer} strength {strength} word {word} done " f"({done}/{total} trials)")

In [ ]:
trials_path = os.path.join(out_dir, "trials.json")
with open(trials_path, "w") as f:
        json.dump(results, f, indent=2)
print(f"[save] {len(results)} trials -> {trials_path}")